In [2]:
import pandas as pd
import torch
from sklearn.utils import shuffle
import pandas as pd

In [3]:
import pandas as pd
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
ratings= pd.read_csv('u.data', sep='\t', names=column_names).iloc[:, 0:3]
ratings

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1
...,...,...,...
99995,880,476,3
99996,716,204,5
99997,276,1090,1
99998,13,225,2


In [4]:
ratings = shuffle(ratings)
ratio = 0.75
# ratings
train = ratings.copy()
test = ratings.copy()
train.iloc[int(ratio*len(ratings)):,2] = None
# train.iloc[int(ratio*len(ratings)):,'rating'] = 0
test.iloc[:int(ratio*len(ratings)),2] = None

In [5]:
train.head()

,user_id,item_id,rating
4772,268,946,3.0
66297,889,678,3.0
28881,521,288,3.0
27123,429,692,3.0
73417,13,619,3.0


In [10]:
rating_matrix = train.pivot(index='user_id', columns='item_id', values='rating')
n_users, n_movies = rating_matrix.shape
# Scaling ratings to between 0 and 1, this helps our model by constraining predictions
min_rating, max_rating = train['rating'].min(), train['rating'].max()
rating_matrix = (rating_matrix - min_rating) / (max_rating - min_rating)
print(rating_matrix)
print(n_users*n_movies-rating_matrix.isnull().values.sum())

item_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                              ...   
1        1.00   NaN  0.75  0.50   0.5   1.0   NaN   NaN   1.0   NaN  ...   
2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.25  ...   
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5        0.75   0.5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
939       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   1.0   NaN  ...   
940       NaN   NaN   NaN  0.25   NaN   NaN  0.75   NaN   NaN   NaN  ...   
941      1.00   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
942       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
943       NaN   1.0   NaN   NaN   NaN   NaN   NaN   NaN   0.5   NaN  ...   

item_id  16

In [12]:
n_users, n_movies

(943, 1682)

In [14]:
# Replacing missing ratings with -1 so we can filter them out later

rating_matrix[rating_matrix.isnull()] = -1
rating_matrix = torch.FloatTensor(rating_matrix.values)
print(rating_matrix)

tensor([[ 1.0000, -1.0000,  0.7500,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        ...,
        [ 1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000,  1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]])


In [16]:
a = torch.ones(3,4)
b = torch.ones(3)

print(b.expand(3, 3))

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


In [18]:
b

tensor([1., 1., 1.])

In [20]:
class SVD(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, u_features, v_features, u_bias, v_bias):
        predicted_rating = torch.sigmoid(torch.mm(u_features, v_features.t())+u_bias.expand(n_movies, n_users).T+(v_bias.expand(n_users, n_movies)))
        
        return predicted_rating

In [22]:
class SVDLoss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3, lam_1 = 0.1, lam_2 = 0.1):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
        self.lam_1 = lam_1
        self.lam_2 = lam_2

    def forward(self, matrix, predicted, u_features, v_features, u_bias, v_bias):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)

        u_regularization = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_regularization = self.lam_v * torch.sum(v_features.norm(dim=1))

        u_bias_regularization = self.lam_1 * (u_bias.norm(dim=0))    
        v_bias_regularization = self.lam_2 * (v_bias.norm(dim=0))    
        return prediction_error + u_regularization + v_regularization + u_bias_regularization + v_bias_regularization

In [24]:
SVD_plus = SVD()
svdloss = SVDLoss(lam_u=0.1, lam_v=0.1, lam_1 = 0.1, lam_2 = 0.1)
num_epoch = 100
latent_vectors = 20

user_features = torch.randn(n_users, latent_vectors, requires_grad=True)
movie_features = torch.randn(n_movies, latent_vectors, requires_grad=True)
user_bias = torch.randn(n_users, requires_grad=True)
movie_bias = torch.randn(n_movies, requires_grad=True)
user_features.data.mul_(0.01)
movie_features.data.mul_(0.01)
user_bias.data.mul_(0.01)
movie_bias.data.mul_(0.01)

optimizer = torch.optim.Adam([user_features, movie_features,user_bias,movie_bias], lr=0.01)
for step, epoch in enumerate(range(num_epoch)):
    optimizer.zero_grad()
    pred = SVD_plus(user_features, movie_features, user_bias, movie_bias)
    loss = svdloss(rating_matrix, pred, user_features, movie_features, user_bias, movie_bias)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"Step {step}, {loss:.3f}")

Step 0, 7253.692
Step 10, 5950.140
Step 20, 4569.626
Step 30, 4081.574
Step 40, 3738.581
Step 50, 3443.907
Step 60, 3136.053
Step 70, 2828.320
Step 80, 2549.618
Step 90, 2319.284


In [25]:
test_rating_matrix = test.pivot(index='user_id', columns='item_id', values='rating')

test_rating_matrix[test_rating_matrix.isnull()] = -1
test_rating_matrix = torch.FloatTensor(test_rating_matrix.values)
test_rating_matrix.shape

torch.Size([943, 1682])

In [26]:
train_items = train["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
test_items = test["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
# Create a mapping from item_id → index in train_items
item_to_index = pd.Series(train_items.index.values, index=train_items.values)

# Map test_items to train indices
test_indices_in_train = test_items.map(item_to_index)

print(test_indices_in_train)

0          0
1          1
2          2
3          3
4          4
        ... 
1677    1677
1678    1678
1679    1679
1680    1680
1681    1681
Name: item_id, Length: 1682, dtype: int64


In [30]:
## test SVD
non_zero_mask = (test_rating_matrix != -1).type(torch.FloatTensor)
num = torch.sum(non_zero_mask)
predicted_ratings = torch.sigmoid(torch.mm(user_features, movie_features.t())+user_bias.expand(n_movies, n_users).T+(movie_bias.expand(n_users, n_movies)))
predicted_ratings = predicted_ratings[:,test_indices_in_train]
predicted_ratings = (predicted_ratings*(max_rating - min_rating) + min_rating)*non_zero_mask
actual_ratings = test_rating_matrix*non_zero_mask

AE_diff = torch.abs(predicted_ratings - actual_ratings)
SE_diff = (predicted_ratings - actual_ratings)**2

test_MAE = torch.sum(AE_diff)/num
test_RMSE = torch.sqrt(torch.sum(SE_diff)/num)

print('test_MAE =', test_MAE.data.numpy())
print('test_RMSE =', test_RMSE.data.numpy())

test_MAE = 0.77042687
test_RMSE = 0.988669
